# OpenCLIP Multi-class Classification
This notebook is used to classify images using OpenCLIP with multiple classes.

## Import necessary libraries

In [1]:
import torch
import os
import json

from PIL import Image
import open_clip
import numpy as np
import pandas as pd

from nazi_symbols_classification.training.data_preparation import get_image_paths
from nazi_symbols_classification.training.evaluation import get_top1_evaluation
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, accuracy_score

/mnt/data/nazi-symbols-classification/venv/lib/python3.12/site-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


## Load the dataset

In [2]:
dir_name = os.path.dirname(os.getcwd())
images = get_image_paths(f"{dir_name}/datasets/nazi-symbols-classification", ("train", "test", "val"))
train_images = [image for image in images if image.startswith(f'{dir_name}/datasets/nazi-symbols-classification/train')]
test_images = [image for image in images if image.startswith(f'{dir_name}/datasets/nazi-symbols-classification/test')]
valid_images = [image for image in images if image.startswith(f'{dir_name}/datasets/nazi-symbols-classification/val')]

In [3]:
y_train = [os.path.basename(os.path.dirname(image)) for image in train_images]
y_test = [os.path.basename(os.path.dirname(image)) for image in test_images]
y_valid = [os.path.basename(os.path.dirname(image)) for image in valid_images]

In [5]:
class_map = {
    'national_rebirth_poland': "neo-nazi", 
    'combat_18_emblem': "neo-nazi", 
    'atomwaffen': "neo-nazi",
    'kolovrat': "neo-nazi", 
    'volksfront_emblem': "neo-nazi", 
    'celtic_cross': "neo-nazi", 
    'hammerskins': "neo-nazi",
    'identitaere_bewegung_emblem': "neo-nazi", 
    'blood_honor_emblem': "neo-nazi", 
    'golden_dawn': "neo-nazi",
    'doppelsiegrune': "siegrune",
}

In [7]:
y_train = [class_map[label] if label in class_map else label for label in y_train ]
y_test = [class_map[label] if label in class_map else label for label in y_test]
y_valid = [class_map[label] if label in class_map else label for label in y_valid]

## Load OpenCLIP model and tokenizer

In [4]:
model, _, preprocess = open_clip.create_model_and_transforms('ViT-B-32', pretrained='laion2b_s34b_b79k')
model.eval()  # model in train mode by default, impacts some models with BatchNorm or stochastic depth active
tokenizer = open_clip.get_tokenizer('ViT-B-32')

## Classify images using OpenCLIP

In [8]:
def classify_image(image_path, prompts):
    text = tokenizer(prompts)
    with torch.no_grad(), torch.autocast("cuda"):
        image = preprocess(Image.open(image_path)).unsqueeze(0)
        image_features = model.encode_image(image)
        text_features = model.encode_text(text)  # type: ignore
        image_features /= image_features.norm(dim=-1, keepdim=True)
        text_features /= text_features.norm(dim=-1, keepdim=True)
        text_probs = (100.0 * image_features @ text_features.T).softmax(dim=-1)
        return {k:round(v.item(), 3) for k, v in dict(zip(prompts, text_probs[0])).items()}

In [15]:
prompts = {
    "A black sun symbol, consisting of concentric circles with radiating, rune-like spokes, associated with Nazi occultism.": "black_sun",
    "The British Union of Fascists logo, a black lightning bolt set within a white circle on a dark background, symbolizing their fascist ideology.": "british_union_of_fascist",
    "A broken sun cross symbol, featuring a circle divided into four or more segments by straight lines, often associated with white supremacist or neo-Nazi groups.": "broken_sun_cross",
    "Historical images of Adolf Hitler addressing crowds, giving speeches, or leading Nazi rallies during the 1930s and 1940s.": "hitler",
    "Images of individuals performing the Hitler salute during historical Nazi Germany events, characterized by a raised right arm held at an angle.": "hitler_salute",
    "Images of the Judenstern, the yellow Star of David badge used during the Holocaust, often featuring the word 'Jude' in black lettering in the center.": "judenstern",
    "Images featuring the 'Happy Merchant' meme, a stereotypical representation of a smiling, hook-nosed Jewish figure used in online racist and antisemitic contexts.": "happy_merchant",
    "Imagery of neo-Nazi groups featuring hate symbols like swastikas, Black Sun, Siegrune, or Celtic Cross on flags, banners, clothing, or graffiti, often seen at rallies, protests, or in propaganda materials promoting white supremacy and far-right ideology.": "neo-nazi",
    "A single angular rune shaped like a lightning bolt or elongated 'S,' used in Nazi and neo-Nazi iconography.": "siegrune",
    "A skull and crossbones insignia, often used by the Nazi SS, with a sinister and militaristic design.": "ss_skull",
    "an image with the sign of sturmabteilung emblem": "sturmabteilung_emblem",
    "A black swastika symbol with arms bent at 90 degrees, typically rotated at a 45-degree angle, often shown on a red circular background or a white circle, used during World War II by Nazi Germany.": "swastika",
    "The Wolfsangel symbol, resembling a hook-like rune, used by Nazi groups and German military units during World War II.": "wolfsangel",
    # "an image containing no nazi related content": "non-nazi",
}

In [10]:
prompts.values()

dict_values(['black_sun', 'british_union_of_fascist', 'broken_sun_cross', 'hitler', 'hitler_salute', 'judenstern', 'happy_merchant', 'neo_nazi', 'siegrune', 'ss_skull', 'sturmabteilung_emblem', 'swastika', 'wolfsangel'])

In [11]:
tmp_result = dict()
classify_result = classify_image(test_images[0], prompts.keys())
for k, v in classify_result.items():
    tmp_result[prompts[k]] = v
sorted(tmp_result.items(), key=lambda x: x[1], reverse=True)[0][0]

'wolfsangel'

## Classify multiple images and save results

In [16]:
%%time

result = []

for image_path in test_images:
    tmp_result = dict()
    classify_result = classify_image(image_path, prompts.keys())
    for k, v in classify_result.items():
        tmp_result[prompts[k]] = v
    result.append(sorted(tmp_result.items(), key=lambda x: x[1], reverse=True)[0][0])

Save the classification results to a JSON file

In [17]:
with open("openclip-output/openclip_result_multiple.json", "w") as f:
    json.dump(result, f)

In [ ]:
with open("openclip-output/openclip_result_multiple.json", "r") as f:
    result = json.load(f)

## Evaluate the classification results

In [18]:
print(classification_report(y_test, result, digits=3))
accuracy_score(y_test, result)

                          precision    recall  f1-score   support

               black_sun      0.960     0.194     0.322       124
british_union_of_fascist      0.261     0.500     0.343        12
        broken_sun_cross      0.026     0.125     0.043        16
          happy_merchant      0.471     1.000     0.641        33
                  hitler      0.610     0.838     0.706       185
           hitler_salute      0.013     0.200     0.024         5
              judenstern      0.125     1.000     0.222         3
                neo-nazi      0.156     0.386     0.222       158
                siegrune      0.508     0.126     0.202       261
                ss_skull      0.503     0.773     0.609       220
   sturmabteilung_emblem      0.000     0.000     0.000         8
                swastika      0.952     0.045     0.086       886
              wolfsangel      0.012     0.182     0.022        33

                accuracy                          0.275      1944
        

0.27469135802469136